<a href="https://colab.research.google.com/github/UPEI-CS/CS1910-Students/blob/main/case_study/Spooky-Authors/CS1910_Spooky-Authors_Workbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CS1910 Computer Science 1: Case Study**
# **Spooky Author Identification**

## Save your own copy first

Only the **computer recorder** needs to do this, but everyone may want a copy afterwards. **Do this before you start**, because anything you type into the original is not saved. Choose:

***File->Save a copy in Drive***

or

***Copy to Drive*** if it appears in the above menu.

Google sign-in is required. You will then be able to open the copy from your Google Drive account later (Colab keeps it in the *Colab Notebooks* folder). At the end, share your copy with the rest of your group (**Share** at the top right).

If you can't use Google Drive, choose ***File->Download .ipynb*** and keep the file by email (signing in to a cloud email service) or on a USB key. To open it in another Colab session later, you would have to move it into your Google Drive account.

## How to use this notebook

This is the coding half of the Spooky Author Identification case study. You work in a **group of 3–5**: the **paper recorder** fills in the group worksheet (`CS1910_Spooky-Authors-Partner_Handout.pdf`), the **computer recorder** types in this notebook, and the **speaker** reports to the class. Plan on paper first, then code: each task says which worksheet stage to do before it.

1. **Run the setup cell** (▶️ below) first. It gives you a `check` helper that tells you ✅ or ❌.
2. **Run the data cells** to download and preview the quotes.
3. **Work through the tasks in order.** For each one, answer the *Understand* questions, write your plan in the ✏️ cell (double-click to edit), then code and run the checks.

Make sure Colab's AI assistance is **off**: this case study is practice with dictionaries, written by your group.

## The authors

- **Mary Wollstonecraft Shelley** (`MWS`), an English novelist, wrote *Frankenstein* (1818).
- **Edgar Allan Poe** (`EAP`), an American writer and poet, wrote *The Raven* (1845) and *The Tell-Tale Heart* (1843).
- **H. P. Lovecraft** (`HPL`), an American horror writer, wrote *The Call of Cthulhu* (1928).

People who know these "spooky" authors well can often tell who wrote a short passage. Our program will try to do the same.

## The approach

1. **Train:** read thousands of quotes whose author we know, and work out how often each author uses each word (the word's **frequency**).
2. **Predict:** for a new quote, multiply together each author's frequencies for its words. The author with the largest product is the guess.
3. **Test:** guess the authors of quotes the program has not seen, and count how many guesses are right.

The approach is simple (and a little naive), and the real goal is practice with dictionaries. The data and the idea come from the [Kaggle Spooky Author Identification](https://www.kaggle.com/c/spooky-author-identification) competition, where you can see more serious approaches.

In [ ]:
#@title ▶️ Run this cell first
#@markdown This cell sets up two helpers, `check` and `check_case`, that tell you
#@markdown whether your answer is right. You do not need to understand how they work.
import math

_cases_passed = {}


def _same(got, expected):
    if isinstance(expected, float) and isinstance(got, (int, float)) and not isinstance(got, bool):
        return math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-12)
    return type(got) == type(expected) and got == expected


def check(got, expected, label="Answer"):
    if _same(got, expected):
        print(f"✅ {label}: {got!r}")
    elif got == expected:
        print(f"❌ {label}: the value looks right, but it is a {type(got).__name__}"
              f" and should be a {type(expected).__name__}")
    else:
        print(f"❌ {label}: got {got!r}, expected {expected!r}")


def check_case(name, case, got, expected_by_case):
    if case not in expected_by_case:
        print(f"ℹ️ {case!r} is not one of the test cases for {name}. Try one of: "
              + ", ".join(repr(c) for c in expected_by_case))
        return
    passed = _cases_passed.setdefault(name, set())
    check(got, expected_by_case[case], f"{name} with {case!r}")
    if _same(got, expected_by_case[case]):
        passed.add(case)
    still = [repr(c) for c in expected_by_case if c not in passed]
    if still:
        print(f"   {len(passed)} of {len(expected_by_case)} test cases passed. "
              f"Change the input and run again. Still to try: {', '.join(still)}")
    else:
        print(f"   🎉 All {len(expected_by_case)} test cases passed!")


print("✅ Practice checker ready.")

## The data

As in most machine-learning work, there are two data sets: `train.csv` (19,479 quotes) to build the model, and `test.csv` (100 different quotes) to see how well it works. Each line of a CSV (comma-separated values) file is one row, with three fields: an id, a quote and the author's initials:

`"id26305","This process, however, afforded me no means of ascertaining the dimensions of my dungeon; ...","EAP"`

Run the next cell to download the data into this Colab session.

In [ ]:
# Download the data files for this case study (run once per Colab session).
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve
        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download("https://github.com/UPEI-CS/CS1910-Students/raw/refs/heads/main/case_study/Spooky-Authors/data.zip")
!unzip -o -q data.zip -x "__MACOSX/*"
print("Data ready: train.csv and test.csv")

## Reading the data with the `csv` module

You could split each line on commas yourself, but quotes contain commas too. The `csv` module handles that. Its `DictReader` turns each row into a **dictionary** with the keys we give it: `"id"`, `"quote"` and `"author"`. Run the cell to see the first two rows.

In [ ]:
import csv

with open("train.csv") as csvfile:
    reader = csv.DictReader(csvfile, fieldnames=["id", "quote", "author"])
    count = 0
    for row in reader:          # each row is a dictionary
        if count < 2:
            print(row)
        count += 1

print("\nrows read:", count)

## Task 1: Understand and plan (worksheet Stages 0–2)

On the worksheet, work through the tiny example by hand, then write your group's plan for training. Come back when your plan has at most 6 steps.

## Task 2: Clean a quote (worksheet Stage 3)

Write `clean_quote(quote, stop_words)`. It returns a **list** of the words in `quote`, after:

1. converting it to lower case;
2. removing every punctuation character (they are all in the string `string.punctuation`);
3. splitting it into words, and keeping only the words that are **not** in `stop_words`.

The stop-word list was suggested by generative AI, including old-fashioned words such as *thou* and *hath*. You may add to it, but your accuracy numbers will then differ from the worksheet's.

**Understand:**
- What should `clean_quote("It was the beating of the old man's heart.", stop_words)` return? (Your worksheet has it.)
- In which order should the steps happen? Does it matter?

✏️ **Your group's plan** (double-click to edit; write it before you code):

1. 
2. 
3. 

In [ ]:
import string

stop_words = [
    "a", "about", "above", "after", "again", "against", "all", "am", "an", "and", "any", "are", "as", "at",
    "be", "because", "been", "before", "being", "below", "between", "both", "but", "by",
    "could",
    "did", "do", "does", "doing", "down", "during",
    "each",
    "few", "for", "from", "further",
    "had", "has", "have", "having", "he", "her", "here", "hers", "herself", "him", "himself", "his", "how",
    "i", "if", "in", "into", "is", "it", "its", "itself",
    "let's",
    "me", "more", "most", "my", "myself",
    "nor", "not",
    "of", "off", "on", "once", "only", "or", "other", "ought", "our", "ours", "ourselves", "out", "over", "own",
    "same", "she", "should", "so", "some", "such",
    "than", "that", "the", "their", "theirs", "them", "themselves", "then", "there", "these", "they", "this", "those", "through", "to", "too",
    "under", "until", "up",
    "very",
    "was", "we", "were", "what", "when", "where", "which", "while", "who", "whom", "why", "with", "would",
    "you", "your", "yours", "yourself", "yourselves",
"thou", "thee", "thy", "thine", "ye", "hath", "doth", "dost", "art", "shalt", "wilt", "hast", "wert",
    "whilst", "ere", "oft", "nay", "yea", "saith", "unto", "wherefore", "whence", "whither"
]


def clean_quote(quote, stop_words):
    """Return the list of words in quote: lower case, no punctuation, no stop words."""
    # Your code here
    pass

# ---- Checks: run the cell to test your code (do not change below) ----
check(clean_quote('You say, "Goodbye" and I say, "Hello, hello, HELLO"!', stop_words),
      ["say", "goodbye", "say", "hello", "hello", "hello"], "the Hello quote")
check(clean_quote("It was the beating of the old man's heart.", stop_words),
      ["beating", "old", "mans", "heart"], "the Tell-Tale Heart quote")
check(clean_quote("And the... THE!", stop_words), [], "only stop words")

## Task 3: Count words and convert to frequencies (worksheet Stage 4)

Write two functions:

- `count_words(good_words, author_dict)` adds 1 to `author_dict[word]` for each word in the list. It **changes** the dictionary it is given and returns nothing. Be careful with a word that is not in the dictionary yet.
- `convert_to_freqs(input_dict)` returns a **new** dictionary with the same keys, where each value is the count divided by the total of all the counts.

**Understand:**
- Why can `count_words` return nothing, and the caller still see the counts? (Think about what a dictionary argument is.)
- `convert_to_freqs` is not given the total. How can you get it from `input_dict`?

✏️ **Your group's plan** (double-click to edit; write it before you code):

1. 
2. 
3. 

In [ ]:
def count_words(good_words, author_dict):
    """Add 1 to author_dict's count for each word in good_words. Returns None."""
    # Your code here
    pass


def convert_to_freqs(input_dict):
    """Return a new dictionary: each word's count divided by the total count."""
    # Your code here
    pass

# ---- Checks: run the cell to test your code (do not change below) ----
counts = {}
count_words(["say", "goodbye", "say", "hello", "hello", "hello"], counts)
check(counts, {"say": 2, "goodbye": 1, "hello": 3}, "counts after one list")
count_words(["hello", "moon"], counts)
check(counts, {"say": 2, "goodbye": 1, "hello": 4, "moon": 1}, "counts after a second list")
check(convert_to_freqs({"say": 2, "goodbye": 1, "hello": 3}), {"say": 2/6, "goodbye": 1/6, "hello": 3/6}, "frequencies")
original = {"a": 1, "b": 3}
convert_to_freqs(original)
check(original, {"a": 1, "b": 3}, "convert_to_freqs does not change its input")

## Train the model

This cell uses your three functions to read all of `train.csv`. It keeps one dictionary of word counts per author, then converts each one into frequencies. You do not need to change it; read it and match each part to your group's plan from Stage 2.

In [ ]:
# Train: count each author's words, then convert the counts to frequencies.
counts_by_author = {"EAP": {}, "MWS": {}, "HPL": {}}

with open("train.csv") as csvfile:
    reader = csv.DictReader(csvfile, fieldnames=["id", "quote", "author"])
    for row in reader:
        good_words = clean_quote(row["quote"], stop_words)
        count_words(good_words, counts_by_author[row["author"]])

freqs_by_author = {}
for author in counts_by_author:
    freqs_by_author[author] = convert_to_freqs(counts_by_author[author])
    print(f"{author}: {len(counts_by_author[author])} different words")

## Task 4: Product of frequencies (worksheet Stage 5)

Write `product_freq(clean_quote, author_freq_dict, depth)`. It returns the product of the frequencies of the words in the quote, using:

- `SMALL_NUMBER` as the frequency of any word that is not in `author_freq_dict`;
- only the `depth` **largest** frequencies (all of them, if the quote has fewer words than `depth`);
- `0.0` for an empty quote.

**Understand:**
- Why not use 0 for an unknown word?
- How can you get the `depth` largest values from a list?

✏️ **Your group's plan** (double-click to edit; write it before you code):

1. 
2. 
3. 

In [ ]:
SMALL_NUMBER = 0.0000001


def product_freq(clean_quote, author_freq_dict, depth):
    """Return the product of the depth largest frequencies of the words in clean_quote."""
    # Your code here
    pass

# ---- Checks: run the cell to test your code (do not change below) ----
quote = ["worst", "episode", "ever"]
author = {"lisa": 0.1, "needs": 0.2, "braces": 0.2, "episode": 0.35, "ever": 0.15}
check(product_freq(quote, author, 1), 0.35, "depth 1")
check(product_freq(quote, author, 2), 0.35 * 0.15, "depth 2")
check(product_freq(quote, author, 3), 0.35 * 0.15 * SMALL_NUMBER, "depth 3 (one unknown word)")
check(product_freq(quote, author, 4), 0.35 * 0.15 * SMALL_NUMBER, "depth 4 (more than the quote)")
check(product_freq([], author, 3), 0.0, "empty quote")

## Test the model

`predict` guesses the author of one quote; `accuracy` runs it over every quote in `test.csv` and returns the fraction it gets right. You do not need to change these. The check confirms that all of your functions work together.

In [ ]:
def predict(quote, depth):
    """Return the author ("EAP", "MWS" or "HPL") whose product is largest for this quote."""
    good_words = clean_quote(quote, stop_words)
    best_author = ""
    best_product = -1.0
    for author in freqs_by_author:
        product = product_freq(good_words, freqs_by_author[author], depth)
        if product > best_product:
            best_author = author
            best_product = product
    return best_author


def accuracy(depth):
    """Return the fraction of test.csv quotes whose author is predicted correctly."""
    correct = 0
    total = 0
    with open("test.csv") as csvfile:
        for row in csv.DictReader(csvfile, fieldnames=["id", "quote", "author"]):
            total += 1
            if predict(row["quote"], depth) == row["author"]:
                correct += 1
    return correct / total


print(f"accuracy with depth 10: {accuracy(10):.0%}")

# ---- Checks: run the cell to test your code (do not change below) ----
check(accuracy(10), 0.81, "accuracy at depth 10")

## Task 5: Experiment (worksheet Stage 6)

Change `depth` and record the accuracy on the worksheet: try 1, 5, 10, 25 and 100. Then discuss: why might looking at more words help, and why does it stop helping?

In [ ]:
for depth in [1, 5, 10, 25, 100]:
    print(f"depth {depth:>3}: {accuracy(depth):.0%}")

## Task 6: Which author is easiest to identify?

Write `accuracy_by_author(depth)`. It returns a dictionary with one key per author, whose value is the fraction of **that author's** test quotes that were predicted correctly. Use `predict`, and two dictionaries of counts.

**Understand:**
- What two things do you need to count for each author?
- Which author's count goes up when a prediction is right: the predicted author, or the real one?

✏️ **Your group's plan** (double-click to edit; write it before you code):

1. 
2. 
3. 

In [ ]:
def accuracy_by_author(depth):
    """Return {author: fraction of that author's test quotes predicted correctly}."""
    # Your code here
    pass

# ---- Checks: run the cell to test your code (do not change below) ----
check(accuracy_by_author(10), {"EAP": 33/40, "HPL": 23/29, "MWS": 25/31}, "accuracy by author, depth 10")

## Before you leave

Tick off each statement your group can honestly agree with. For any you can't, go back to the matching task or worksheet stage, and bring your questions to the next class.

- [ ] I can use a dictionary to count things, including keys seen for the first time (`.get(key, 0)`).
- [ ] I can turn counts into frequencies, and explain why a function can change a dictionary it is given.
- [ ] I can explain why the text is cleaned and why unknown words get `SMALL_NUMBER`.
- [ ] I can explain the difference between training data and test data.

**Share your copy** of this notebook with the rest of your group before you go.

### Further reading

This is a simple version of ideas from **natural language processing**. Real systems use measures such as *cosine similarity* and *TF-IDF* (term frequency–inverse document frequency) to compare word distributions. Accuracy is not always the best measure either: read about precision and recall if you are curious.